# Four by four

In [ ]:
import json

import geopandas as gpd
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import requests
import us

from census import Census
from shapely.geometry import Point
import numpy as np
import pandas as pd

from pyei.two_by_two import TwoByTwoEI
from pyei.goodmans_er import GoodmansER
from pyei.goodmans_er import GoodmansERBayes
from pyei.r_by_c import RowByColumnEI



states_fips = [
    "06", # California
    "25", # Massachusetts
    "26", # Michigan
    "36", # New York
    "48" # Texas
] #these are the states whose files we got in ertotalrun.ipynb

state_dfs = ['0']*len(states_fips)

for i in range(len(states_fips)):
    #df = pd.read_csv('ecologicalcsvs/' + states_fips[i] + 'ecological_acs_pl.csv')
    state_dfs[i] = pd.read_csv('ecologicalcsvs/' + states_fips[i] + 'ecological_acs_pl_723.csv')

for i in range(len(state_dfs)):
    state_dfs[i].fillna(0, inplace = True) #the EI won't run otherwise; inplace = True means don't return anything


In [ ]:
def two_by_four(election_dataframe):
    # Replace with the column names in your data that give the percentages of demographic groups of interest
    # These fractions must sum to 1 within each precinct.
    # candidates are race and demographics are the ancestries. 
    group_fractions_rbyc = np.array(election_dataframe[['white_ACS_pct', 'black_ACS_pct', 'sor_ACS_pct', 'not_sor0_w0_b0_ACS_pct']]).T
    
    # Replace with the column names in your data that give the votes for candidates of interest
    # These fractions  must sum to 1 within each precinct.
    # Make sure to use to correct denominator - it should match what you use for precinct_pops below
    votes_fractions_rbyc = np.array(election_dataframe[['brazilian_ACS_pct', 'non_brazilian_ACS_pct']]).T
    
    # replace 'total2' with the column name for the total count of votes in your data
    # this should be equal to the sum of the counts of votes for the candidates of interest above!
    precinct_pops = np.array(election_dataframe['totpop_ACS']).astype(int)
    
    candidate_names_rbyc = ["Brazilian", "Non-Brazilian"] #votes fractions
    demographic_group_names_rbyc = ["SOR0", "W0", "B0", "Other"] # group fractions
    
    # Create a RowByColumnEI object
    ei_rbyc = RowByColumnEI(model_name='multinomial-dirichlet')
    
    # Fit the model
    ei_rbyc.fit(
           group_fractions_rbyc,
           votes_fractions_rbyc,
           precinct_pops,
           demographic_group_names=demographic_group_names_rbyc,
           candidate_names=candidate_names_rbyc,
           chains=4
    )
    print(ei_rbyc.summary())
    return ei_rbyc

In [ ]:
def er_two_by_four(ei_rbyc):
    ei_rbyc.plot_kdes(plot_by="candidate")

In [ ]:
for i in range(len(state_dfs)):
    state_dfs[i]['non_brazilian_ACS_pct'] = 1- state_dfs[i]['brazilian_ACS_pct']
    state_dfs[i]['not_sor0_w0_b0_ACS_pct'] = 1-state_dfs[i]['white_ACS_pct'] - state_dfs[i]['black_ACS_pct'] - state_dfs[i]['sor_ACS_pct']
    inference = two_by_four(state_dfs[i])
    er_two_by_four(inference)